In [1]:
# Session Master Table
import os
import sqlite3
import pandas as pd

In [2]:
# load cleaned tables
df_session = pd.read_csv(r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\cleaned_data\session_cleaned.csv')
df_user    = pd.read_csv(r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\cleaned_data\user_cleaned.csv')
df_order   = pd.read_csv(r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\cleaned_data\order_cleaned.csv')

In [3]:
# load into sqlite — renaming order to orders to avoid sql reserved word conflict
conn = sqlite3.connect(':memory:')
df_session.to_sql('sessions', conn, index=False, if_exists='replace')
df_user.to_sql('users',       conn, index=False, if_exists='replace')
df_order.to_sql('orders',     conn, index=False, if_exists='replace')

11206

In [ ]:
query = """
SELECT
    s.session_id,
    s.user_id,
    s.time,
    s.platform,
    s.device_type,
    s.country,
    s.referrer,
    s.landing_page,
    s.browser,
    s.utm_source,
    s.utm_medium,
    s.utm_campaign,

    -- user info
    u.has_purchase_last_year,
    u.has_purchase_last_qtr,
    u.user_segment,

    -- marketing channel bucket
   CASE
    WHEN s.utm_medium = 'cpc'         THEN 'Paid Search'
    WHEN s.utm_medium = 'paid_social' THEN 'Paid Social'
    WHEN s.utm_medium = 'display'     THEN 'Display'
    WHEN s.utm_medium = 'affiliate'   THEN 'Affiliate'
    WHEN s.utm_medium = 'email'       THEN 'Email'
    WHEN s.utm_medium = 'sms'         THEN 'SMS'
    WHEN s.utm_medium = 'internal'    THEN 'Internal'
    WHEN s.utm_medium = 'organic'     THEN 'Organic'
    WHEN s.referrer   = 'direct'      THEN 'Direct'
    ELSE 'Other'
END AS channel

    -- converted or not
    CASE
        WHEN o.session_id IS NOT NULL THEN 1
        ELSE 0
    END AS is_converted,

    -- order details if converted
    o.total_price,
    o.discount,
    o.net_revenue,
    o.has_coupon

FROM sessions s
LEFT JOIN users  u ON s.user_id    = u.user_id
LEFT JOIN orders o ON s.session_id = o.session_id
"""



In [5]:
session_master = pd.read_sql_query(query, conn)
conn.close()

In [6]:
# quick check
print(session_master.shape)
print(session_master['is_converted'].value_counts())
print(session_master['channel'].value_counts())
display(session_master.head(10))

# save
MASTER = r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\master_tables'
os.makedirs(MASTER, exist_ok=True)
session_master.to_csv(os.path.join(MASTER, 'session_master.csv'), index=False)
print("saved → session_master.csv")

(146000, 21)
is_converted
0    134794
1     11206
Name: count, dtype: int64
channel
Paid Social    55930
Email          22537
Paid Search    11397
Display        11299
SMS            11298
Affiliate      11211
Direct         11181
Internal       11147
Name: count, dtype: int64


,session_id,user_id,time,platform,device_type,country,referrer,landing_page,browser,utm_source,...,utm_campaign,has_purchase_last_year,has_purchase_last_qtr,user_segment,channel,is_converted,total_price,discount,net_revenue,has_coupon
0,a32f32ca-14ed-44d5-afb4-070d082e88e7,9d415f62-5d38-436d-a443-336b2b759568,2025-01-01 23:17:00,web,desktop,us,impact.com,/products/video-doorbell-pro-2,chrome,impact,...,holiday_sale,0,0,new_or_inactive,Affiliate,0,NaN,NaN,NaN,NaN
1,f43c11f7-9553-4408-ae10-98fd285ba383,a71251a4-f215-4046-85fb-51206780e6c8,2025-01-01 22:41:00,web,desktop,us,ring.com,/products/stick-up-cam-battery,safari,ring.com,...,remarketing,1,1,active,Internal,0,NaN,NaN,NaN,NaN
2,14d9b0a5-d40c-43aa-9f02-c4a24f3f0c3a,a8e69945-e3f7-4458-8ea9-883062d75c71,2025-01-01 12:06:00,web,desktop,us,linkedin.com,/products/video-doorbell-pro-2,chrome,linkedin,...,none,0,0,new_or_inactive,Paid Social,0,NaN,NaN,NaN,NaN
3,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,a71251a4-f215-4046-85fb-51206780e6c8,2025-01-01 02:13:00,web,desktop,us,link.shortener,/products/ring-alarm-8-piece,safari,tiktok,...,launch_q4,1,1,active,Paid Social,1,749.97,20.0,729.97,1.0
4,74c787bd-d697-47a2-9d17-ed47415fae0c,d2280282-85c3-40fe-93b8-990479b73fb3,2025-01-01 23:59:00,web,desktop,india,direct,/products/stick-up-cam-battery,safari,tiktok,...,none,0,0,new_or_inactive,Paid Social,0,NaN,NaN,NaN,NaN
5,2fe302a4-25f2-4b66-9834-7c9e137cfe77,a71251a4-f215-4046-85fb-51206780e6c8,2025-01-01 00:24:00,web,mobile,us,direct,/products/indoor-cam-(2nd-gen),safari,organic,...,remarketing,1,1,active,Direct,0,NaN,NaN,NaN,NaN
6,95bb95ee-2a4a-4eee-bac1-c6fbdbf82eab,19755aa9-50fc-44f2-80c0-e2eb4ecb4160,2025-01-01 23:20:00,web,mobile,us,google.com,/products/video-doorbell-pro-2,edge,dv360,...,none,1,1,active,Display,0,NaN,NaN,NaN,NaN
7,5c7c20e5-5703-4f6f-b1ad-80a475412961,b632f58c-f467-4363-9b87-b01c237df412,2025-01-01 18:38:00,web,desktop,us,direct,/products/stick-up-cam-battery,chrome,tiktok,...,brand,0,0,new_or_inactive,Paid Social,0,NaN,NaN,NaN,NaN
8,e52bfbe0-9774-4642-a262-04170c77858a,ff30683f-bd0b-4ada-9cc8-547049fdc62c,2025-01-01 17:00:00,web,mobile,india,facebook.com,/products/indoor-cam-(2nd-gen),safari,facebook,...,holiday_sale,0,0,new_or_inactive,Paid Social,0,NaN,NaN,NaN,NaN
9,2c6ee273-f443-4ec2-a378-b44f18a41a4e,3cc6932b-7cc1-4852-b6a8-3348607240eb,2025-01-01 23:10:00,web,mobile,us,facebook.com,/products/indoor-cam-(2nd-gen),edge,facebook,...,launch_q4,0,0,new_or_inactive,Paid Social,0,NaN,NaN,NaN,NaN


saved → session_master.csv


In [7]:
print(session_master['is_converted'].value_counts())
print(session_master['is_converted'].mean().round(4) * 100, '% conversion rate')

is_converted
0    134794
1     11206
Name: count, dtype: int64
7.68 % conversion rate


In [8]:
# check converted sessions have order values filled
converted = session_master[session_master['is_converted'] == 1]
print(converted[['total_price', 'discount', 'net_revenue', 'has_coupon']].isnull().sum())
print(converted.shape)

total_price    0
discount       0
net_revenue    0
has_coupon     0
dtype: int64
(11206, 21)
